# RTMLib Pose Estimation - Google Colab GPU ワーカー起動

このノートブックを実行すると、Google Colab 上で `gui.py --persistent-worker` を常駐させた FastAPI サーバーと Cloudflare Quick Tunnel が起動します。

1. ランタイムの種類を **GPU（T4 / L4 / A100）** に変更
2. セル 1〜4 を上から順に実行
3. セル 4 に表示される **公開 URL (`https://....trycloudflare.com`)** と **API トークン** をローカルアプリの「Colab 接続設定」へ貼り付け

In [ ]:
#@title 1. GPU確認 & 基本設定
REPO_URL = "https://github.com/Shiro-to215/rtmlib-pose-estimation.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}
USE_DRIVE_CACHE = True  #@param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/rtmlib_pose_cache"  #@param {type:"string"}
REPO_DIR = "/content/rtmlib-pose-estimation"

!nvidia-smi
import os
print(f"CPU数: {os.cpu_count()}")

In [ ]:
#@title 2. コード取得 & 依存パッケージ・cloudflared 導入
import os
import subprocess
from pathlib import Path

clone_url = REPO_URL
if GITHUB_TOKEN.strip() and REPO_URL.startswith("https://"):
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN.strip()}@", 1)

repo_path = Path(REPO_DIR)
if (repo_path / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_path), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url, str(repo_path)], check=True)

%cd /content/rtmlib-pose-estimation
subprocess.run(["pip", "install", "-q", "-r", "colab_server/requirements-colab.txt"], check=True)
# CPU版onnxruntimeが混入しないよう rtmlib は --no-deps で導入する
subprocess.run(["pip", "install", "-q", "--no-deps", "rtmlib==0.0.16"], check=True)

if not Path("/usr/local/bin/cloudflared").is_file():
    subprocess.run([
        "curl", "-fsSL", "-o", "/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    os.chmod("/usr/local/bin/cloudflared", 0o755)
print("セットアップ完了")

In [ ]:
#@title 3. 高速ローカルキャッシュ設定 & 任意 Drive 同期
import os
import sys
from pathlib import Path

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

LOCAL_CACHE_DIR = Path("/content/.cache")
os.environ["VITPOSE_ONNX_CACHE_DIR"] = str(LOCAL_CACHE_DIR / "vitpose_onnx")
os.environ["HF_HOME"] = str(LOCAL_CACHE_DIR / "huggingface")
os.environ["YOLO_CONFIG_DIR"] = str(LOCAL_CACHE_DIR / "ultralytics")
os.environ["TORCH_HOME"] = str(LOCAL_CACHE_DIR / "torch")

for env_key in ("VITPOSE_ONNX_CACHE_DIR", "HF_HOME", "YOLO_CONFIG_DIR", "TORCH_HOME"):
    Path(os.environ[env_key]).mkdir(parents=True, exist_ok=True)

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    from colab_server.cache_sync import sync_drive_to_local
    stats = sync_drive_to_local(DRIVE_CACHE_DIR, LOCAL_CACHE_DIR)
    print(f"Drive -> ローカルSSD同期結果: {stats}")
else:
    print("Driveキャッシュ同期: OFF (/content/.cache のみ使用)")

In [ ]:
#@title 4. APIサーバー & Cloudflare トンネル起動（終了時は停止ボタンで停止）
import os
import re
import secrets
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

API_TOKEN = secrets.token_urlsafe(24)
os.environ["COLAB_API_TOKEN"] = API_TOKEN
os.environ["COLAB_WORK_DIR"] = "/content/colab_work"
os.environ["COLAB_LOCAL_CACHE_DIR"] = "/content/.cache"
os.environ["COLAB_DRIVE_CACHE_DIR"] = DRIVE_CACHE_DIR if USE_DRIVE_CACHE else ""

server_proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "colab_server.app"],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# /health が応答するまで待機（初回は常駐ワーカーの起動も同時に実施）
for _ in range(90):
    if server_proc.poll() is not None:
        out = server_proc.stdout.read() if server_proc.stdout else ""
        raise RuntimeError(f"APIサーバー起動失敗:\n{out}")
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8000/health",
            headers={"Authorization": f"Bearer {API_TOKEN}"},
        )
        with urllib.request.urlopen(req, timeout=2):
            break
    except Exception:
        time.sleep(1)

tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
assert tunnel_proc.stdout is not None
for line in tunnel_proc.stdout:
    m = url_re.search(line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    server_proc.terminate()
    tunnel_proc.terminate()
    raise RuntimeError("Cloudflare トンネルURLの取得に失敗しました。")

print("=" * 64)
print("ローカルアプリの「Colab 接続設定」へ以下をコピーして貼り付けてください:")
print(f"  サーバー URL : {public_url}")
print(f"  API トークン : {API_TOKEN}")
print("=" * 64)

try:
    while server_proc.poll() is None and tunnel_proc.poll() is None:
        time.sleep(5)
except KeyboardInterrupt:
    print("停止リクエストを受信しました...")
finally:
    for p in (tunnel_proc, server_proc):
        if p.poll() is None:
            p.terminate()
            try:
                p.wait(timeout=5)
            except subprocess.TimeoutExpired:
                p.kill()
    if USE_DRIVE_CACHE:
        from colab_server.cache_sync import sync_local_to_drive
        stats = sync_local_to_drive("/content/.cache", DRIVE_CACHE_DIR)
        print(f"終了時 Drive 同期結果: {stats}")